# Pipeline completo — microbioma cervicovaginal e câncer cervical

**Validação por projeto em meta-análises de microbioma (LATINOWARE / Latin.Science, 2026).**
Autora: Tassiane Anzolin · PPGTCA / UTFPR.

Este caderno reúne, em um único lugar e na ordem de execução, **todos os passos** do pipeline próprio,
do levantamento de metadados aos quatro testes estatísticos finais, sobre 10 BioProjects públicos
(997 amostras catalogadas → 889 finais). Cada etapa traz a explicação e os comandos reais executados.

> As células de código são o registro exato dos comandos (bash do QIIME 2, R e Python). Não são
> feitas para rodar de cima a baixo num kernel só: o pipeline mistura shell, QIIME 2 e R, e depende
> de vários GB de FASTQ. O valor aqui é a reprodutibilidade passo a passo.

## 1. Accessions e metadados clínicos

Mapeamento Run/Sample → classe clínica (câncer, HSIL, LSIL, controle) para as 1.042 amostras
catalogadas, a partir de `metadata.txt` (compartilhado pela coautora Paola Gyuliane Gonçalves).
A contagem por projeto e classe bate célula a célula com a Tabela I do artigo.

In [ ]:
# metadata.txt — colunas: #NAME(run);sample_accession;tipo_amostra;HPV;Center;Location;Project
# Contagem por projeto e classe (conferida):
#   P1 : cancer=68 control=4        P2 : cancer=10 hsil=27 lsil=11 control=29
#   P6 : cancer=26                  P9 : cancer=58
#   P10: cancer=1 hsil=59 lsil=43 control=216
#   P14: hsil=22 control=21         P16: cancer=55
#   P19: cancer=31 hsil=60 lsil=58 control=19
#   P21: hsil=2 control=103         P22: cancer=65 control=54
#   TOTAL: cancer=314 hsil=170 lsil=112 control=446  (1042)

## 2. Baixar os FASTQ brutos

Um projeto por vez, via SRA Toolkit, organizados por código de projeto (P1..P22). A região do gene
(V3-V4 ou V4) muda entre projetos e condiciona toda a etapa seguinte.

In [ ]:
mkdir -p raw/P1 && cd raw/P1
prefetch --option-file ../../accessions/P1_runs.txt
fasterq-dump --split-files --threads 4 SRR*/*.sra
gzip *.fastq
cd ../..

## 3. Import, corte de primers e preparo para o deblur

Import no QIIME 2 (detecção pareado/single por amostra), corte de primers com cutadapt, merge de
pares e filtro de qualidade.

**Achados da execução real:** P9, P14 e P2 já vieram sem primer (retenção ~0% ao tentar cortar) →
cópia direta. P19 corrigido empiricamente para 341F/805R (retenção 55% → 83,7%). P10 é **misto**:
~154 amostras com primer 515F(Caporaso) e ~165 já sem primer (tratadas como P10b). P21 sem primer
publicado → excluído.

In [ ]:
# Import (10 projetos; detecta pareado/single por amostra)
python3 generate_manifests.py --raw-dir raw --out-dir manifests
conda activate rachis-qiime2-2026.7
./import_all.sh   # gera qiime/{proj}_paired.qza e/ou _single.qza + qiime/import_log.tsv

In [ ]:
# Grupo 338F/806R (V3-V4): P1, P6
qiime cutadapt trim-paired \
  --i-demultiplexed-sequences qiime/P1_paired.qza \
  --p-front-f ACTCCTACGGGAGGCAGCA --p-front-r GGACTACHVGGGTWTCTAAT \
  --p-discard-untrimmed \
  --o-trimmed-sequences qiime/P1_trimmed.qza --o-stats qiime/P1_cutadapt_stats.qza
# repetir para P6 (mesmo par)

In [ ]:
# 515F(Parada)/806R, V4: P16
qiime cutadapt trim-paired --i-demultiplexed-sequences qiime/P16_paired.qza \
  --p-front-f GTGYCAGCMGCCGCGGTAA --p-front-r GGACTACNVGGGTWTCTAAT \
  --p-discard-untrimmed --o-trimmed-sequences qiime/P16_trimmed.qza --o-stats qiime/P16_cutadapt_stats.qza

# P10: 515F(Caporaso, sem Y), V4, single-end
qiime cutadapt trim-single --i-demultiplexed-sequences qiime/P10_single.qza \
  --p-front GTGCCAGCMGCCGCGGTAA --p-discard-untrimmed \
  --o-trimmed-sequences qiime/P10_trimmed.qza --o-stats qiime/P10_cutadapt_stats.qza

# P19: 341F/805R (corrigido; fonte antiga indicava 357wF/806R)
qiime cutadapt trim-paired --i-demultiplexed-sequences qiime/P19_paired.qza \
  --p-front-f CCTACGGGNGGCWGCAG --p-front-r GACTACHVGGGTATCTAATCC \
  --p-discard-untrimmed --o-trimmed-sequences qiime/P19_trimmed.qza --o-stats qiime/P19_cutadapt_stats.qza

# P22: 341F, single-end
qiime cutadapt trim-single --i-demultiplexed-sequences qiime/P22_single.qza \
  --p-front CCTAYGGGRBGCASCAG --p-discard-untrimmed \
  --o-trimmed-sequences qiime/P22_trimmed.qza --o-stats qiime/P22_cutadapt_stats.qza

In [ ]:
# P9, P14, P2: primer ja removido na origem -> copia direta
cp qiime/P9_paired.qza  qiime/P9_trimmed.qza
cp qiime/P14_paired.qza qiime/P14_trimmed.qza
cp qiime/P2_paired.qza  qiime/P2_trimmed.qza
# P21: SKIP (sequencia proprietaria Novogene, sem par de referencia)
echo "P21 excluido: sem primer confirmado" >> primers_skip.log

In [ ]:
# Merge de pares (projetos ainda pareados) e filtro de qualidade
for p in P1 P6 P9 P14 P16 P19 P2; do
  qiime vsearch merge-pairs --i-demultiplexed-seqs qiime/${p}_trimmed.qza \
    --o-merged-sequences qiime/${p}_joined.qza --o-unmerged-sequences qiime/${p}_unmerged.qza
done
for p in P1 P6 P9 P14 P16 P19 P2; do
  qiime quality-filter q-score --i-demux qiime/${p}_joined.qza \
    --o-filtered-sequences qiime/${p}_filtered.qza --o-filter-stats qiime/${p}_filter_stats.qza
done
for p in P10 P22; do  # ja single-end
  qiime quality-filter q-score --i-demux qiime/${p}_trimmed.qza \
    --o-filtered-sequences qiime/${p}_filtered.qza --o-filter-stats qiime/${p}_filter_stats.qza
done

## 4. Denoising com deblur (por região)

Cada região (V3-V4 e V4) recebe um lote de deblur separado, com seu próprio comprimento de corte.
Antes, os `*_filtered.qza` de cada região viram um único artefato (via export → manifest combinado →
reimport, porque `merge-seqs` exige `FeatureData[Sequence]`, não `SampleData[...]`).

**Retenção real:** V3-V4 `--p-trim-length 250` → 289/309 (93,5%). V4 `--p-trim-length 220` → 627/628
(99,8%), depois de corrigir o manifest duplicado do P10 (sublote P10b).

In [ ]:
mkdir -p export/V3V4 export/V4
for p in P1 P6 P14 P19; do qiime tools export --input-path qiime/${p}_filtered.qza --output-path export/V3V4/${p}; done
for p in P2 P9 P10 P16 P22; do qiime tools export --input-path qiime/${p}_filtered.qza --output-path export/V4/${p}; done

In [ ]:
import glob, os, csv
for regiao, projetos in [("V3V4", ["P1","P6","P14","P19"]), ("V4", ["P2","P9","P10","P16","P22"])]:
    rows = []
    for p in projetos:
        for fp in sorted(glob.glob(f"export/{regiao}/{p}/*.fastq.gz")):
            sample_id = os.path.basename(fp).split(".")[0].split("_")[0]
            rows.append((sample_id, os.path.abspath(fp)))
    out = f"manifests/{regiao}_combined_manifest.csv"
    with open(out, "w", newline="") as f:
        w = csv.writer(f, delimiter="\t", lineterminator="\n")
        w.writerow(["sample-id", "absolute-filepath"]); w.writerows(rows)
    print(out, len(rows), "amostras")

In [ ]:
# Isola as 165 IDs do sublote sem primer do P10 (percent-reads-after < 10%)
awk -F'\t' 'NR>2 && $3<10 {print $1}' /tmp/p10_cutadapt_stats/metadata.tsv > /tmp/p10_sem_primer.txt
# importa P10b direto (sem cutadapt) e filtra qualidade
qiime tools import --type 'SampleData[SequencesWithQuality]' \
  --input-path manifests/P10_sem_primer_manifest.csv --input-format SingleEndFastqManifestPhred33V2 \
  --output-path qiime/P10_sem_primer.qza
qiime quality-filter q-score --i-demux qiime/P10_sem_primer.qza \
  --o-filtered-sequences qiime/P10_sem_primer_filtered.qza --o-filter-stats qiime/P10_sem_primer_filter_stats.qza
# o manifest combinado da V4 exclui essas 165 IDs do P10a antes de juntar com P10b (evita ID duplicada)

In [ ]:
qiime tools import --type 'SampleData[SequencesWithQuality]' \
  --input-path manifests/V3V4_combined_manifest.csv --input-format SingleEndFastqManifestPhred33V2 \
  --output-path qiime/V3V4_filtered_merged.qza
qiime tools import --type 'SampleData[SequencesWithQuality]' \
  --input-path manifests/V4_combined_manifest.csv --input-format SingleEndFastqManifestPhred33V2 \
  --output-path qiime/V4_filtered_merged.qza

qiime deblur denoise-16S --i-demultiplexed-seqs qiime/V3V4_filtered_merged.qza --p-trim-length 250 \
  --p-sample-stats --o-representative-sequences qiime/V3V4_repseqs.qza \
  --o-table qiime/V3V4_table.qza --o-stats qiime/V3V4_stats.qza
qiime deblur denoise-16S --i-demultiplexed-seqs qiime/V4_filtered_merged.qza --p-trim-length 220 \
  --p-sample-stats --o-representative-sequences qiime/V4_repseqs.qza \
  --o-table qiime/V4_table.qza --o-stats qiime/V4_stats.qza

![Retenção do deblur por região](../analysis/figuras/f12_contabilidade_amostras.png)

## 5. Atribuição taxonômica (SILVA)

O classificador pré-treinado oficial foi recusado por incompatibilidade de versão do scikit-learn
(artefato 0.24.1 × ambiente 1.7.1). Solução: baixar as referências brutas do SILVA 138, extrair a
região de cada par de primer e **retreinar** um Naive Bayes por região (1 a 5 h cada).

In [ ]:
curl -sSL -o silva-138-99-seqs.qza "https://data.qiime2.org/2024.2/common/silva-138-99-seqs.qza"
curl -sSL -o silva-138-99-tax.qza  "https://data.qiime2.org/2024.2/common/silva-138-99-tax.qza"

qiime feature-classifier extract-reads --i-sequences silva-138-99-seqs.qza \
  --p-f-primer ACTCCTACGGGAGGCAGCA --p-r-primer GGACTACHVGGGTWTCTAAT --p-n-jobs 4 \
  --o-reads silva-138-99-V3V4-seqs.qza --o-read-extraction-stats silva-138-99-V3V4-extraction-stats.qza
qiime feature-classifier extract-reads --i-sequences silva-138-99-seqs.qza \
  --p-f-primer GTGYCAGCMGCCGCGGTAA --p-r-primer GGACTACNVGGGTWTCTAAT --p-n-jobs 4 \
  --o-reads silva-138-99-V4-seqs.qza --o-read-extraction-stats silva-138-99-V4-extraction-stats.qza

qiime feature-classifier fit-classifier-naive-bayes --i-reference-reads silva-138-99-V3V4-seqs.qza \
  --i-reference-taxonomy silva-138-99-tax.qza --o-classifier silva-138-99-V3V4-classifier.qza
qiime feature-classifier fit-classifier-naive-bayes --i-reference-reads silva-138-99-V4-seqs.qza \
  --i-reference-taxonomy silva-138-99-tax.qza --o-classifier silva-138-99-V4-classifier.qza

qiime feature-classifier classify-sklearn --i-classifier silva-138-99-V3V4-classifier.qza \
  --i-reads qiime/V3V4_repseqs.qza --p-n-jobs 4 --o-classification qiime/V3V4_taxonomy.qza
qiime feature-classifier classify-sklearn --i-classifier silva-138-99-V4-classifier.qza \
  --i-reads qiime/V4_repseqs.qza --p-n-jobs 4 --o-classification qiime/V4_taxonomy.qza

## 6. Filtragem e fusão por gênero

Remove cloroplasto/mitocôndria/arqueia e singletons; colapsa cada região em **gênero** (nível 6 do
SILVA) para tornar V3-V4 e V4 comparáveis; mescla e filtra por profundidade (≥1.000 leituras).
**Resultado: 889 amostras finais** (de um teto de 937 sem o P21; 89,2% das 997 publicadas).

In [ ]:
for r in V3V4 V4; do
  qiime taxa filter-table --i-table qiime/${r}_table.qza --i-taxonomy qiime/${r}_taxonomy.qza \
    --p-exclude mitochondria,chloroplast,archaea --o-filtered-table qiime/${r}_table_clean.qza
  qiime feature-table filter-features --i-table qiime/${r}_table_clean.qza --p-min-frequency 2 \
    --o-filtered-table qiime/${r}_table_nosingle.qza
  qiime taxa collapse --i-table qiime/${r}_table_nosingle.qza --i-taxonomy qiime/${r}_taxonomy.qza \
    --p-level 6 --o-collapsed-table qiime/${r}_genus.qza
done
qiime feature-table merge --i-tables qiime/V3V4_genus.qza qiime/V4_genus.qza \
  --p-overlap-method sum --o-merged-table qiime/genus_table_merged.qza
qiime feature-table filter-samples --i-table qiime/genus_table_merged.qza \
  --p-min-frequency 1000 --o-filtered-table qiime/genus_table_final.qza   # 889 amostras

## 7. Partição de variância (PERMANOVA, R)

Bray-Curtis, termos condição clínica + localidade + projeto, sobre as 889 amostras finais.
**Condição explica 0,59%; projeto explica 9,19% (≈15,5×). Localidade colinear com projeto (0 g.l.).**

In [ ]:
library(vegan)
tabela <- read.delim("export/genus_table_final/feature-table.tsv", skip=1, header=TRUE,
                     row.names=1, check.names=FALSE); tabela_t <- t(tabela)
metadata <- read.delim("metadata_final_889.tsv", header=TRUE, stringsAsFactors=FALSE)
rownames(metadata) <- metadata$sampleid; metadata <- metadata[rownames(tabela_t), ]
dist_bc <- vegdist(tabela_t, method="bray")
adonis2(dist_bc ~ tipo_amostra + Location + Project, data=metadata, permutations=999, by="margin")
#              Df SumOfSqs      R2       F Pr(>F)
# tipo_amostra  3     2.14 0.00593  2.2311  0.002 **
# Project       5    33.18 0.09193 20.7474  0.001 ***

In [ ]:
# Contrastes par a par (controla o confundimento ao restringir a 2 classes)
pares <- list(c("control","cancer"), c("lesion high","cancer"),
              c("lesion low","cancer"), c("lesion high","lesion low"))
for (p in pares) {
  idx <- metadata$tipo_amostra %in% p
  print(adonis2(as.dist(dist_bc_matriz[idx,idx]) ~ tipo_amostra,
                data=metadata[idx,], permutations=999))
}
# control vs cancer 6,52% (n=624) | HSIL vs cancer 5,75% (n=454)
# LSIL vs cancer 4,44% (n=409)    | HSIL vs LSIL n.s. (p=0,107)

![PERMANOVA](../analysis/figuras/f07_permanova.png)

## 8. Ordenação por variáveis latentes (GLLVM, R)

98 gêneros presentes em ≥10% das amostras. AIC cai de 313.373 (sem covariáveis) para 300.307,4
(com projeto). Os dois eixos latentes são dominados por **projeto** (η² 56,7% e 58,2%) e só
marginalmente por condição clínica (11,1% e 7,9%).

In [ ]:
library(gllvm)
prevalencia <- colSums(tabela_t > 0) / nrow(tabela_t)
tabela_filtrada <- as.matrix(tabela_t[, prevalencia >= 0.10])
metadata$Project <- factor(metadata$Project)
modelo_sem <- gllvm(y=tabela_filtrada, family="negative.binomial", num.lv=2)
modelo_com <- gllvm(y=tabela_filtrada, X=metadata[,"Project",drop=FALSE],
                    formula=~Project, family="negative.binomial", num.lv=2)
escores <- getLV(modelo_sem)
d <- data.frame(LV1=escores[,1], LV2=escores[,2],
                tipo_amostra=factor(metadata$tipo_amostra), Project=metadata$Project)
summary(aov(LV1 ~ Project, data=d)); summary(aov(LV1 ~ tipo_amostra, data=d))
# LV1~Project eta2=56,7% | LV1~tipo_amostra eta2=11,1%  (AIC 313373 -> 300307.4)

![GLLVM](../analysis/figuras/f08_gllvm.png)

## 9. Protocolo de validação (Random Forest, R)

A contribuição central do artigo: CV estratificada aleatória × leave-one-project-out (LOPO), mais um
controle negativo que prevê o **projeto** a partir da composição. Abundância relativa por gênero.

**Macro-AUC cai de 0,811 (CV aleatória) para 0,6 (LOPO)** — e para **0,457** com a direção da ROC fixa
e o P22 reprocessado. Controle negativo prevê o projeto com **82,8%** de acurácia (acaso ≈ 11%).

In [ ]:
library(ranger); library(pROC)
tabela_rel <- tabela_t / rowSums(tabela_t)
# Esquema 1: CV estratificada aleatoria (10 folds)
for (k in 1:10) rf <- ranger(x=tabela_rel[folds_random!=k,], y=metadata$tipo_amostra[folds_random!=k], probability=TRUE)
# Esquema 2: leave-one-project-out
for (p in levels(metadata$Project)) rf <- ranger(x=tabela_rel[metadata$Project!=p,], y=metadata$tipo_amostra[metadata$Project!=p], probability=TRUE)
# Controle negativo: prever o projeto
for (k in 1:10) rf <- ranger(x=tabela_rel[folds_proj!=k,], y=metadata$Project[folds_proj!=k])
# CV aleatoria: cancer .959 control .815 hsil .760 lsil .710  | macro-AUC 0.811
# LOPO:         cancer .717 control .512 hsil .510 lsil .662  | macro-AUC 0.6 (0.457 com direcao fixa)
# Controle negativo: acuracia 0.8279

![AUC por validação](../analysis/figuras/f01_auc_por_validacao.png)

![Prever o projeto](../analysis/figuras/f05_prever_projeto.png)

## Fechamento

As quatro etapas estatísticas (PERMANOVA, GLLVM, Random Forest e contrastes par a par) estão
concluídas sobre as 889 amostras do pipeline próprio. O padrão é consistente e independente:
**o projeto de origem domina a variação composicional muito mais que a condição clínica**, e o
desempenho de classificação despenca sob validação leave-one-project-out. É o mesmo problema de
confundimento que a dissertação (soja) vai enfrentar, agora documentado de ponta a ponta.